In [1]:
import pandas as pd
import numpy as np

Reading Standard pitcher from the 2024 and 2025 season

In [43]:
# Initializing empty dataframe for standard pitcher stats
standard_pitcher = pd.DataFrame()
years = ['2023', '2024', '2025']
prediction_year = 2024

#loop through years and concatinate into one dataframe
for year in years:
  path = f"/content/standard_pitcher_stats_{year}.csv"

  stats = pd.read_csv(path)
  stats['Year'] = int(year)
  standard_pitcher = pd.concat([standard_pitcher, stats])

# Excluding pitchers with less than 60 IP
standard_pitcher = standard_pitcher[standard_pitcher['IP'] >= 60]
# Filtering to only pitchers with 3 years of data
def three_years(df):
  return len(df) == 3

standard_pitcher = standard_pitcher.groupby('PlayerId').filter(three_years)

# Filtering columns (A lot of extra columns which will not be used)
standard_pitcher = standard_pitcher[['Name', 'Team', 'Year', 'W', 'L', 'ERA', 'G',
    'GS', 'QS', 'IP', 'H', 'ER', 'HR', 'BB', 'SO', 'PlayerId']]

# Deriving additional features
standard_pitcher['per_9'] = round(standard_pitcher['IP'] / 9.0, 2)
standard_pitcher['K_per_9'] = round(standard_pitcher['SO'] / standard_pitcher['per_9'], 4)
standard_pitcher['BB_per_9'] = round(standard_pitcher['BB'] / standard_pitcher['per_9'], 4)
standard_pitcher['H_per_9'] = round(standard_pitcher['H'] / standard_pitcher['per_9'], 4)
standard_pitcher['HR_per_9'] = round(standard_pitcher['HR'] / standard_pitcher['per_9'], 4)
standard_pitcher.drop(columns=['per_9'], inplace=True)
standard_pitcher['K_to_BB'] = standard_pitcher.apply(lambda row: round(row['SO'] / row['BB'], 2) if row['BB'] != 0 else row['SO'], axis = 1)

# Handling case where relievers snuck through the IP filter
standard_pitcher['Quality_Start_percent'] = standard_pitcher.apply(lambda row: row['QS'] / row['GS'] if row['GS'] != 0 else None, axis = 1)
standard_pitcher['Quality_Start_percent'] = round(standard_pitcher['Quality_Start_percent'], 4) * 100

# Binning Pitcher ERAs 0 - 3: Elite, 3 - 3.99: Good, 4 - 4.99: Average, 5+: Poor
bins = [0, 3, 3.99, 4.99, np.inf]
labels = [0, 1, 2, 3]
#labels = ['Elite', 'Good', 'Average', 'Poor']

standard_pitcher['ERA_label'] = pd.cut(standard_pitcher['ERA'], bins=bins, labels=labels, include_lowest=True)
standard_pitcher['ERA_label'] = standard_pitcher['ERA_label'].astype('int')
# Building lag features
standard_pitcher = standard_pitcher.sort_values(by=['Year', 'PlayerId'])
standard_pitcher['YoY_ERA'] = standard_pitcher.groupby('PlayerId')['ERA'].diff()
standard_pitcher['YoY_K_per_9'] = standard_pitcher.groupby('PlayerId')['K_per_9'].diff()
standard_pitcher['YoY_BB_per_9'] = standard_pitcher.groupby('PlayerId')['BB_per_9'].diff()

# Setting the target variable to next years ERA
standard_pitcher['Target'] = standard_pitcher.groupby('PlayerId')['ERA'].shift(-1)

# Dropping all features not needed for ERA prediction
features = ['W', 'L', 'H', 'ER', 'HR', 'BB', 'SO', 'QS']
standard_pitcher.drop(columns=features, inplace=True)

# Filtering to only years
standard_pitcher = standard_pitcher[(standard_pitcher['Year'] == prediction_year) & ~(standard_pitcher['Quality_Start_percent'].isna())]


Reading plus pitcher stats from 2023 - 2024

In [44]:
plus_stats = pd.DataFrame()
years = ['2023', '2024']

for year in years:
  path = f"/content/plus_pitcher_stats_{year}.csv"

  stats = pd.read_csv(path)
  stats['Year'] = int(year)

  plus_stats = pd.concat([plus_stats, stats])

# Excluding pitchers with less than 60 innings
plus_stats = plus_stats[plus_stats['IP'] >= 60]

# Dropping extra columns
plus_stats.drop(columns=['IP', 'K/9+', 'K/BB+', 'BB/9+', 'HR/9+', 'AVG+', 'WHIP+', 'LD%+', 'FB%+', 'NameASCII', 'MLBAMID', 'Name', 'Team'], inplace=True)

# Filtering to only players with 2 years of data
def two_years(df):
  return len(df) == 2

plus_stats = plus_stats.groupby('PlayerId').filter(two_years)

plus_stats = plus_stats.sort_values(['Year', 'PlayerId'])

# Adding lag features
plus_stats['YoY_ERA_minus'] = plus_stats.groupby('PlayerId')['ERA-'].diff()
plus_stats['YoY_FIP_minus'] = plus_stats.groupby('PlayerId')['FIP-'].diff()
plus_stats['YoY_K_plus_pct'] = plus_stats.groupby('PlayerId')['K%+'].diff()

plus_stats = plus_stats[plus_stats['Year'] == 2024]


Reading StatCast Pitcher Stats 2023-2024

In [45]:
statcast_stats = pd.DataFrame()
years = ['2023', '2024']

for year in years:
  path = f"/content/statcast_pitcher_stats_{year}.csv"

  stats = pd.read_csv(path)
  stats['Year'] = int(year)

  statcast_stats = pd.concat([statcast_stats, stats])

# Dropping pitchers with less than 60 IP
statcast_stats = statcast_stats[statcast_stats['IP'] >= 60]

# Dropping extra columns
statcast_stats.drop(columns = ['Name', 'Team', 'IP', 'Events', 'Barrels', 'HardHit', 'EV90', 'maxEV', 'LA', 'ERA', 'NameASCII', 'MLBAMID'], inplace = True)
# Filtering out pitchers without 2 years of data
def two_years(df):
  return len(df) == 2

statcast_stats = statcast_stats.groupby('PlayerId').filter(two_years)

# Creating lag features
statcast_stats = statcast_stats.sort_values(by=['PlayerId', 'Year'])

statcast_stats['YoY_Barrel_pct'] = statcast_stats.groupby('PlayerId')['Barrel%'].diff()
statcast_stats['YoY_xERA'] = statcast_stats.groupby('PlayerId')['xERA'].diff()
statcast_stats['YoY_HardHit_pct'] = statcast_stats.groupby('PlayerId')['HardHit%'].diff()

statcast_stats = statcast_stats[statcast_stats['Year'] == 2024]


Merging all Tables together (Left join since standard_pitcher holds the Target variable)


In [46]:
final_table = standard_pitcher.copy()

final_table = pd.merge(final_table, plus_stats, on=['PlayerId', 'Year'], how = 'left')

final_table = pd.merge(final_table, statcast_stats, on=['PlayerId', 'Year'], how = 'left')

Writing Final DataFrame to CSV for future use

In [48]:
final_table.to_csv('pitcher_stats_table.csv')